<a href="https://colab.research.google.com/github/eco-bios/monitoreo-deforestacion-chiapas/blob/main/Pipeline_Deforestacion_Chiapas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este Script permite autenticar la cuenta e inicializa la conexion con el proyecto y se verifica mediante un dataset la conexion a los servidores de google earth

In [14]:
import ee
import warnings
# Esto ignora los mensajes de advertencia que no son críticos
warnings.filterwarnings('ignore')

# --- AUTENTICACIÓN ---
# Inicia el proceso de autorización para acceder a la cuenta de Earth Engine.
ee.Authenticate()

try:
    # --- INICIALIZACIÓN ---
    # Intenta establecer conexión con el proyecto específico en Google Cloud
    ee.Initialize(project='monitoreo-deforestacion-ch')

    # --- PRUEBA DE CONEXIÓN ---
    # Se carga un dataset global (SRTM) para verificar el acceso a los servidores
    test_image = ee.Image("USGS/SRTMGL1_003")
    print("✅ ¡CONECTADO! El motor de Google Earth Engine ya te reconoce.")

except Exception as e:
    # Mensaje en caso de fallo en las credenciales o el ID del proyecto
    print(f"❌ Error detallado: {e}")

✅ ¡CONECTADO! El motor de Google Earth Engine ya te reconoce.


Esta función automatiza la obtención de salud vegetal (NDVI) comparando dos periodos. Aplica filtros de nubosidad y promedia los valores en un radio de 1 km para asegurar resultados representativos a nivel de paisaje. Ademas puede utilizarse en distintos periodos y de forma escalable.

In [19]:
# --- LA MÁQUINA DE PROCESAMIENTO (FUNCIÓN) ---
def analizar_salud_vegetal(lat, lon, nombre_sitio):
    # 1. Crear geometría del punto de análisis
    punto = ee.Geometry.Point([lon, lat])

    # 2. Función interna: Filtra colección Sentinel-2 y calcula NDVI (B8 y B4)
    def get_ndvi_anual(fecha_inicio, fecha_fin):
        coleccion = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(punto)
            .filterDate(fecha_inicio, fecha_fin)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))) # Filtro de nubes
        return coleccion.median().normalizedDifference(['B8', 'B4'])

    # 3. Procesamiento de periodos: Línea base (2024) y Actualidad (2026)
    ndvi_2024 = get_ndvi_anual('2024-01-01', '2024-01-31')
    ndvi_2026 = get_ndvi_anual('2026-01-01', '2026-01-31')

    # 4. Reducción estadística: Promedio en un radio de 1000m (buffer)
    # Usamos .get('nd') porque normalizedDifference nombra la banda como 'nd'
    val_24 = ndvi_2024.reduceRegion(ee.Reducer.mean(), punto.buffer(1000), 10).get('nd').getInfo()
    val_26 = ndvi_2026.reduceRegion(ee.Reducer.mean(), punto.buffer(1000), 10).get('nd').getInfo()

    # 5. Salida de resultados y cálculo de variación (Delta)
    if val_24 and val_26:
        dif = val_26 - val_24
        print(f"📍 {nombre_sitio} | 2024: {val_24:.3f} | 2026: {val_26:.3f} | Delta: {dif:+.3f}")
    else:
        print(f"❌ Sin datos suficientes en {nombre_sitio}")

En esta sección se invoca la función para diferentes ubicaciones de interés en Chiapas. Gracias a la modularidad del código, podemos monitorear ecosistemas variados (desde humedales hasta selva) simplemente actualizando las coordenadas decimales.

In [21]:
# --- PANEL DE CONTROL: AQUÍ CAMBIAS LAS COORDENADAS ---

# 1. San Juan Panamá (Región del Soconusco - Escuintla)
analizar_salud_vegetal(15.29, -92.60, "San Juan Panamá")

# 2. Catazajá (Región de Humedales y pastizales)
analizar_salud_vegetal(17.72, -91.71, "Catazajá")

# 3. Ocosingo (Zona de transición a la Selva Lacandona)
analizar_salud_vegetal(16.90, -92.09, "Ocosingo")

📍 San Juan Panamá | 2024: 0.676 | 2026: 0.775 | Delta: +0.098
📍 Catazajá | 2024: 0.457 | 2026: 0.481 | Delta: +0.023
📍 Ocosingo | 2024: 0.411 | 2026: 0.424 | Delta: +0.013


Generamos un mapa dinámico para inspeccionar visualmente la salud vegetal. Utilizamos la librería geemap para superponer el ráster de NDVI sobre la ubicación analizada, permitiendo una validación visual de los datos calculados.

In [23]:
# --- CONFIGURACIÓN DEL MAPA ---

# 1. Definimos el área de interés para el mapa (San Juan Panamá)
lat, lon = 15.29, -92.60
punto_vista = ee.Geometry.Point([lon, lat])

# 2. Inicialización del Mapa Interactivo
Map = geemap.Map(center=[lat, lon], zoom=13)

# 3. Parámetros de Visualización (Semáforo de salud vegetal)
parametros_vis = {
    'min': 0,
    'max': 1,
    'palette': ['#e74c3c', '#f1c40f', '#2ecc71'] # Rojo (bajo) a Verde (alto)
}

# 4. Capas: Se usa el ndvi_2026 generado previamente por la función
# Nota: Asegúrate de haber corrido la función antes de esta celda.
Map.addLayer(ndvi_2026, parametros_vis, 'Salud Vegetal (Enero 2026)')
Map.addLayer(punto_vista, {'color': '#3498db'}, 'Ubicación: San Juan Panamá')

# 5. Despliegue
Map

Map(center=[15.29, -92.6], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

Exportación de Resultados a Google Drive
Para asegurar la persistencia de los datos y permitir análisis posteriores en software SIG externo (como QGIS), exportamos el ráster de NDVI procesado. Se define una región rectangular que abarca el área de estudio y se utiliza el formato Cloud Optimized GeoTIFF para optimizar el rendimiento del archivo.

In [ ]:
# 1. Definición del área de exportación
# Generamos un área rectangular (bounding box) de 2km x 2km para el archivo de salida.
geometria_exportar = punto_escuintla.buffer(2000).bounds()

# 2. Configuración de la tarea de exportación a Google Drive
tarea = ee.batch.Export.image.toDrive(
    image=ndvi_2026,
    description='NDVI_SanJuanPanama_2026',
    folder='Proyecto_Monitoreo_Chiapas',
    scale=10,                      # Resolución espacial de Sentinel-2
    region=geometria_exportar,
    fileFormat='GeoTIFF',
    formatOptions={'cloudOptimized': True}
)

# 3. Ejecución y monitoreo de la tarea
# La tarea se procesa en los servidores de Google Earth Engine de forma asíncrona.
print("🚀 Iniciando exportación a Google Drive...")
tarea.start()

import time
while tarea.active():
    print(f"⌛ Estado de la tarea: {tarea.status()['state']}...", end="\r")
    time.sleep(15)

print(f"\n✅ Proceso completado. Archivo disponible en la carpeta 'Proyecto_Monitoreo_Chiapas'.")